# Model Training

This notebook shows the generalized sklearn-style model training workflow. It assumes the weekly model feature table has already been built by the data pipeline or `04_feature-engineering.ipynb`.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression

from gas_forecast.data.features import DEFAULT_WEATHER_MODEL_FEATURES, TARGET_COLUMN
from gas_forecast.data.paths import latest_processed_path
from gas_forecast.modeling import ExpandingWindowSplitter, HoldoutSplitter, run_backtest
from gas_forecast.plotting import plot_weekly_change_forecast

In [ ]:
REGION = "R48"
PROCESSED_DIR = Path("../datasets/processed")
FEATURES_PATH = latest_processed_path(REGION, "weekly_model_features", PROCESSED_DIR)

features = pd.read_parquet(FEATURES_PATH)
feature_cols = list(DEFAULT_WEATHER_MODEL_FEATURES)
features[["date", TARGET_COLUMN, *feature_cols]].tail()

## Holdout Backtest

The holdout splitter gives one train/validation split. This matches the older notebook habit of comparing model output against a single recent year or period.

In [ ]:
holdout = HoldoutSplitter(
    date_col="date",
    train_end="2024-12-31",
    val_start="2025-01-01",
)

linear_predictions, linear_metrics = run_backtest(
    features,
    feature_cols=feature_cols,
    target_col=TARGET_COLUMN,
    date_col="date",
    model=LinearRegression(),
    splitter=holdout,
)

linear_metrics

In [ ]:
plot_weekly_change_forecast(
    linear_predictions,
    model_name="Linear Regression Holdout",
).show()

## Expanding-Window Backtest

The same `run_backtest` function works with a different validation strategy and a different sklearn estimator.

In [ ]:
expanding = ExpandingWindowSplitter(
    date_col="date",
    initial_train_start="2010-01-01",
    initial_train_end="2020-12-31",
    val_weeks=52,
    step_weeks=52,
)

tree_predictions, tree_metrics = run_backtest(
    features,
    feature_cols=feature_cols,
    target_col=TARGET_COLUMN,
    date_col="date",
    model=HistGradientBoostingRegressor(random_state=42),
    splitter=expanding,
)

tree_metrics

In [ ]:
plot_weekly_change_forecast(
    tree_predictions,
    model_name="HistGradientBoosting Expanding CV",
).show()